# Apache Hudi - Time Travel & Audit

Apache Hudi keeps a timeline of commits, which makes it possible to query historical versions of a table and build an audit trail for record changes.
Unlike plain Parquet datasets, Hudi stores metadata about write operations, commit instants, record keys, partitions, and file versions.

## What you will learn

In this notebook, you will learn:

- Creating multiple commits in a Hudi Copy-On-Write (COW) table
- Understanding Hudi commit instants
- Querying the latest version of a record
- Using time travel with `as.of.instant`
- Building a simple audit view from Hudi metadata columns
- Comparing record versions across commits
- Inspecting table history without relying on previous notebooks

## Step 1 — Create Spark session

This notebook assumes that the Hudi bundle JAR is already available in the Spark Docker image under `/opt/spark/jars`.

Because of that, we do not use `spark.jars.packages` here. This avoids Ivy/Maven downloads from inside the notebook.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Hudi-06-Time-Travel-Audit")
    .master("spark://spark-master:7077")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.sql.extensions", "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.hudi.catalog.HoodieCatalog")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 4.0.2


## Step 2 — Define shared paths and table name

This notebook is independent and creates its own table. It does not require any table from previous notebooks.

In [2]:
BASE_PATH = "/workspace/data/hudi"
TABLE_NAME = "riders_time_travel_audit"
TABLE_PATH = f"{BASE_PATH}/{TABLE_NAME}"

print("Table name:", TABLE_NAME)
print("Table path:", TABLE_PATH)

Table name: riders_time_travel_audit
Table path: /workspace/data/hudi/riders_time_travel_audit


## Step 3 — Clean previous run

For a tutorial notebook, we remove the previous table path so that reruns start from a known state.

In [3]:
import shutil
import os

if os.path.exists(TABLE_PATH):
    shutil.rmtree(TABLE_PATH)
    print(f"Removed existing table path: {TABLE_PATH}")
else:
    print("No existing table path found.")

No existing table path found.


## Step 4 — Define Hudi write options

The table uses:

- `rider` as the record key
- `city` as the partition path
- `ts` as the precombine field
- `COPY_ON_WRITE` table type for simple versioned storage

In [4]:
hudi_write_options = {
    "hoodie.table.name": TABLE_NAME,
    "hoodie.datasource.write.table.name": TABLE_NAME,
    "hoodie.datasource.write.recordkey.field": "rider",
    "hoodie.datasource.write.partitionpath.field": "city",
    "hoodie.datasource.write.precombine.field": "ts",
    "hoodie.datasource.write.table.type": "COPY_ON_WRITE",
    "hoodie.datasource.write.operation": "upsert",
}

## Step 5 — Create multiple commits

We write three updates for the same rider. Each write creates a new Hudi commit instant.

The `ts` column is the business timestamp. The Hudi commit instant is stored separately in `_hoodie_commit_time`.

In [5]:
from pyspark.sql.functions import to_timestamp
import time

updates = [
    ("r5", "SEA", "2024-01-07 10:00:00"),
    ("r5", "SEA", "2024-01-07 11:00:00"),
    ("r5", "SEA", "2024-01-07 12:00:00"),
]

for index, row in enumerate(updates):
    df = spark.createDataFrame([row], ["rider", "city", "ts"]).withColumn("ts", to_timestamp("ts"))

    write_mode = "overwrite" if index == 0 else "append"

    (
        df.write
        .format("hudi")
        .options(**hudi_write_options)
        .mode(write_mode)
        .save(TABLE_PATH)
    )

    print(f"Written commit candidate for business timestamp: {row[2]}")
    time.sleep(1)

print("All writes completed.")

# WARNING: Unable to get Instrumentation. Dynamic Attach failed. You may add this JAR as -javaagent manually, or supply -Djdk.attach.allowAttachSelf
# WARNING: Unable to attach Serviceability Agent. You can try again with escalated privileges. Two options: a) use -Djol.tryWithSudo=true to try with sudo; b) echo 0 | sudo tee /proc/sys/kernel/yama/ptrace_scope


[Stage 33:>                                                         (0 + 1) / 1]

26/04/25 19:46:46 WARN HoodieTableFileSystemView: Partition: SEA is not available in store


Written commit candidate for business timestamp: 2024-01-07 10:00:00


Written commit candidate for business timestamp: 2024-01-07 11:00:00


Written commit candidate for business timestamp: 2024-01-07 12:00:00
All writes completed.


## Step 6 — Read the latest table version

A normal Hudi read returns the latest snapshot of the table.

In [6]:
latest_df = spark.read.format("hudi").load(TABLE_PATH)
latest_df.createOrReplaceTempView("riders_audit_latest")

spark.sql("""
SELECT
    rider,
    city,
    ts,
    _hoodie_commit_time,
    _hoodie_record_key,
    _hoodie_partition_path,
    _hoodie_file_name
FROM riders_audit_latest
ORDER BY rider
""").show(truncate=False)

+-----+----+-------------------+-------------------+------------------+----------------------+-------------------------------------------------------------------------+
|rider|city|ts                 |_hoodie_commit_time|_hoodie_record_key|_hoodie_partition_path|_hoodie_file_name                                                        |
+-----+----+-------------------+-------------------+------------------+----------------------+-------------------------------------------------------------------------+
|r5   |SEA |2024-01-07 12:00:00|20260425194709325  |r5                |SEA                   |852d283f-b364-42e8-9420-1f45d805b860-0_0-79-512_20260425194709325.parquet|
+-----+----+-------------------+-------------------+------------------+----------------------+-------------------------------------------------------------------------+



## Step 7 — Capture commit instants

For time travel, we use Hudi commit instants from `_hoodie_commit_time`.

This is safer than reading timeline files directly from `.hoodie`, because timeline file names may differ between Hudi versions.

In [7]:
commits = [
    row["_hoodie_commit_time"]
    for row in latest_df
        .select("_hoodie_commit_time")
        .distinct()
        .orderBy("_hoodie_commit_time")
        .collect()
]

if not commits:
    raise RuntimeError("No Hudi commit instants found. Check that the write steps completed successfully.")

print("Commit instants visible in the latest snapshot:")
for commit in commits:
    print(commit)

first_commit = commits[0]
latest_commit = commits[-1]

print(f"\nFirst visible commit instant: {first_commit}")
print(f"Latest visible commit instant: {latest_commit}")

Commit instants visible in the latest snapshot:
20260425194709325

First visible commit instant: 20260425194709325
Latest visible commit instant: 20260425194709325


## Step 8 — Read historical versions with incremental query

A latest snapshot shows only the newest version of a record.

To build an audit trail of all changes, use Hudi incremental query mode. This reads records written between commit instants.

In [8]:
incremental_df = (
    spark.read
    .format("hudi")
    .option("hoodie.datasource.query.type", "incremental")
    .option("hoodie.datasource.read.begin.instanttime", "000")
    .load(TABLE_PATH)
)

incremental_df.createOrReplaceTempView("riders_audit_incremental")

spark.sql("""
SELECT
    rider,
    city,
    ts,
    _hoodie_commit_time,
    _hoodie_record_key,
    _hoodie_partition_path
FROM riders_audit_incremental
ORDER BY _hoodie_commit_time
""").show(truncate=False)

26/04/25 19:47:33 WARN CacheManager: Asked to cache already cached data.
+-----+----+-------------------+-------------------+------------------+----------------------+
|rider|city|ts                 |_hoodie_commit_time|_hoodie_record_key|_hoodie_partition_path|
+-----+----+-------------------+-------------------+------------------+----------------------+
|r5   |SEA |2024-01-07 12:00:00|20260425194709325  |r5                |SEA                   |
+-----+----+-------------------+-------------------+------------------+----------------------+



## Step 9 — Time travel to the first visible commit

This reads the table as of the first visible commit instant.

In [9]:
spark.read.format("hudi") \
    .option("as.of.instant", first_commit) \
    .load(TABLE_PATH) \
    .createOrReplaceTempView("riders_as_of_first_commit")

spark.sql("""
SELECT
    rider,
    city,
    ts,
    _hoodie_commit_time
FROM riders_as_of_first_commit
WHERE rider = 'r5'
""").show(truncate=False)

26/04/25 19:47:39 WARN CacheManager: Asked to cache already cached data.
26/04/25 19:47:41 WARN CacheManager: Asked to cache already cached data.
+-----+----+-------------------+-------------------+
|rider|city|ts                 |_hoodie_commit_time|
+-----+----+-------------------+-------------------+
|r5   |SEA |2024-01-07 12:00:00|20260425194709325  |
+-----+----+-------------------+-------------------+



## Step 10 — Time travel to the latest commit

This reads the table as of the latest visible commit instant.

In [10]:
spark.read.format("hudi") \
    .option("as.of.instant", latest_commit) \
    .load(TABLE_PATH) \
    .createOrReplaceTempView("riders_as_of_latest_commit")

spark.sql("""
SELECT
    rider,
    city,
    ts,
    _hoodie_commit_time
FROM riders_as_of_latest_commit
WHERE rider = 'r5'
""").show(truncate=False)

26/04/25 19:47:46 WARN CacheManager: Asked to cache already cached data.
26/04/25 19:47:48 WARN CacheManager: Asked to cache already cached data.
+-----+----+-------------------+-------------------+
|rider|city|ts                 |_hoodie_commit_time|
+-----+----+-------------------+-------------------+
|r5   |SEA |2024-01-07 12:00:00|20260425194709325  |
+-----+----+-------------------+-------------------+



## Step 11 — Build a simple audit trail

The audit trail shows every version of the record captured by the incremental query.

In [11]:
spark.sql("""
SELECT
    rider,
    city,
    ts AS business_timestamp,
    _hoodie_commit_time AS hudi_commit_instant,
    _hoodie_file_name AS physical_file
FROM riders_audit_incremental
WHERE rider = 'r5'
ORDER BY hudi_commit_instant
""").show(truncate=False)

+-----+----+-------------------+-------------------+-------------------------------------------------------------------------+
|rider|city|business_timestamp |hudi_commit_instant|physical_file                                                            |
+-----+----+-------------------+-------------------+-------------------------------------------------------------------------+
|r5   |SEA |2024-01-07 12:00:00|20260425194709325  |852d283f-b364-42e8-9420-1f45d805b860-0_0-79-512_20260425194709325.parquet|
+-----+----+-------------------+-------------------+-------------------------------------------------------------------------+



## Step 12 — Compare business time and commit time

The business timestamp (`ts`) comes from the source data.

The Hudi commit instant (`_hoodie_commit_time`) is generated by Hudi when the write is committed.

These two values answer different questions:

- `ts`: When did the event happen in the business domain?
- `_hoodie_commit_time`: When did Hudi commit this version to the table?

In [12]:
spark.sql("""
SELECT
    rider,
    ts AS business_timestamp,
    _hoodie_commit_time AS hudi_commit_instant
FROM riders_audit_incremental
WHERE rider = 'r5'
ORDER BY hudi_commit_instant
""").show(truncate=False)

+-----+-------------------+-------------------+
|rider|business_timestamp |hudi_commit_instant|
+-----+-------------------+-------------------+
|r5   |2024-01-07 12:00:00|20260425194709325  |
+-----+-------------------+-------------------+



## Step 13 — Inspect table files

This step lists files under the table path so you can see the partition folders and Hudi metadata area.

In [13]:
from pathlib import Path

table_path = Path(TABLE_PATH)

print("Table files:")
for path in sorted(table_path.rglob("*")):
    if path.is_file():
        print(path.relative_to(table_path))

Table files:
.hoodie/.hoodie.properties.crc
.hoodie/.index_defs/.index.json.crc
.hoodie/.index_defs/index.json
.hoodie/hoodie.properties
.hoodie/metadata/.hoodie/.hoodie.properties.crc
.hoodie/metadata/.hoodie/hoodie.properties
.hoodie/metadata/.hoodie/timeline/.00000000000000000.deltacommit.inflight.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000000.deltacommit.requested.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000000_20260425194630990.deltacommit.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000001.deltacommit.inflight.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000001.deltacommit.requested.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000001_20260425194633281.deltacommit.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000002.deltacommit.inflight.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000002.deltacommit.requested.crc
.hoodie/metadata/.hoodie/timeline/.00000000000000002_20260425194637421.deltacommit.crc
.hoodie/metadata/.hoodie/timeline/.20260

## Step 14 — Summary

In this notebook, you created an independent Hudi table with multiple commits and used it for time travel and audit inspection.

Key takeaways:

- Hudi stores commit metadata with each record.
- Snapshot reads show the latest table state.
- Time travel reads let you query older table versions.
- Incremental queries are useful for building audit trails.
- Business timestamps and Hudi commit instants are different concepts.